In [1]:
import numpy as np
import pandas as pd
from scapy.all import rdpcap, raw
import matplotlib.pyplot as plt
import seaborn as sns



𝗔𝘁𝘂𝗿 𝗝𝗮𝗹𝘂𝗿 𝗣𝗮𝗸𝗲𝘁 𝗱𝗮𝗻 𝗽𝗮𝗻𝗴𝗴𝗶𝗹 𝗱𝗮𝘁𝗮𝘀𝗲𝘁

In [2]:
import pandas as pd
from scapy.all import rdpcap
from collections import Counter
import numpy as np

# =========================
# KONFIGURASI PATH
# =========================
BASE_PATH = '/home/dani/Documents/tugas akhir/TugasAkhir/DataTugasAkhirku2026/dataTOWIDSmentah/'

TRAIN_PCAP_PATH = BASE_PATH + 'Automotive_Ethernet_with_Attack_original_10_17_19_50_training.pcap'
TRAIN_LABELS_PATH = BASE_PATH + 'y_train.csv'

TEST_PCAP_PATH = BASE_PATH + 'Automotive_Ethernet_with_Attack_original_10_17_20_04_test.pcap'
TEST_LABELS_PATH = BASE_PATH + 'y_test.csv'

print("Mulai load PCAP...")

# =========================
# LOAD PCAP
# =========================
train_packets = rdpcap(TRAIN_PCAP_PATH)
test_packets = rdpcap(TEST_PCAP_PATH)

print(f"Jumlah paket TRAIN : {len(train_packets)}")
print(f"Jumlah paket TEST  : {len(test_packets)}")


# =========================
# FUNGSI EKSTRAK FULL PAYLOAD
# TANPA POTONG PANJANG PAKET
# =========================
def extract_packet_lengths(packets):
    packet_lengths = []
    
    for pkt in packets:
        raw_bytes = bytes(pkt)   # ambil seluruh isi paket
        packet_lengths.append(len(raw_bytes))
    
    return packet_lengths


# =========================
# EKSTRAK PANJANG PAKET
# =========================
train_lengths = extract_packet_lengths(train_packets)
test_lengths = extract_packet_lengths(test_packets)

# Gabungkan supaya analisis global
all_lengths = train_lengths + test_lengths

print(f"\nTotal paket dianalisis: {len(all_lengths)}")


# =========================
# CARI PANJANG TERPANJANG
# =========================
max_length = max(all_lengths)

# =========================
# CARI PANJANG TERPENDEK
# =========================
min_length = min(all_lengths)

# =========================
# NILAI TENGAH (MIDPOINT)
# bukan median
# =========================
middle_value = (max_length + min_length) / 2

# =========================
# MEDIAN PANJANG PAKET
# =========================
median_length = np.median(all_lengths)

# =========================
# PANJANG YANG PALING SERING MUNCUL
# =========================
length_counter = Counter(all_lengths)

most_common_length, frequency = length_counter.most_common(1)[0]


Mulai load PCAP...
Jumlah paket TRAIN : 1203737
Jumlah paket TEST  : 791611

Total paket dianalisis: 1995348


In [3]:
# =========================
# TAMPILKAN JUMLAH LABEL
# F_I, P_I, M_F, C_D, C_R
# PADA DATASET TRAIN DAN TEST
# =========================

# Load label CSV
y_train = pd.read_csv(TRAIN_LABELS_PATH)
y_test = pd.read_csv(TEST_LABELS_PATH)

print("Kolom pada y_train:", y_train.columns.tolist())
print("Kolom pada y_test :", y_test.columns.tolist())

# Ambil kolom label
# Jika CSV hanya punya satu kolom, gunakan kolom tersebut.
# Jika lebih dari satu kolom, gunakan kolom terakhir.
train_label_col = y_train.columns[-1]
test_label_col = y_test.columns[-1]

train_labels = y_train[train_label_col].astype(str).str.strip()
test_labels = y_test[test_label_col].astype(str).str.strip()

# Daftar kelas yang ingin ditampilkan
target_labels = ['F_I', 'P_I', 'M_F', 'C_D', 'C_R']

# Hitung jumlah masing-masing label
train_counts = train_labels.value_counts()
test_counts = test_labels.value_counts()

# Buat tabel perbandingan
label_summary = pd.DataFrame({
    'Label': target_labels,
    'Jumlah Train': [train_counts.get(label, 0) for label in target_labels],
    'Jumlah Test': [test_counts.get(label, 0) for label in target_labels]
})

# Tambahkan total per kelas
label_summary['Total'] = (
    label_summary['Jumlah Train'] +
    label_summary['Jumlah Test']
)

print("\n==============================")
print("JUMLAH LABEL TRAIN DAN TEST")
print("==============================")
display(label_summary)

# Total seluruh data yang memiliki label target
print("\nTotal paket label target pada TRAIN:", label_summary['Jumlah Train'].sum())
print("Total paket label target pada TEST :", label_summary['Jumlah Test'].sum())
print("Total paket label target          :", label_summary['Total'].sum())

# Opsional: tampilkan semua label yang tersedia di masing-masing dataset
print("\n==============================")
print("SEMUA DISTRIBUSI LABEL TRAIN")
print("==============================")
display(train_labels.value_counts().rename_axis('Label').reset_index(name='Jumlah'))

print("\n==============================")
print("SEMUA DISTRIBUSI LABEL TEST")
print("==============================")
display(test_labels.value_counts().rename_axis('Label').reset_index(name='Jumlah'))

Kolom pada y_train: ['e1', 'Normal', 'Normal.1']
Kolom pada y_test : ['1', 'Normal', 'Normal.1']

JUMLAH LABEL TRAIN DAN TEST


,Label,Jumlah Train,Jumlah Test,Total
0,F_I,35112,16962,52074
1,P_I,64635,26013,90648
2,M_F,33765,16809,50574
3,C_D,85466,41203,126669
4,C_R,29847,29847,59694



Total paket label target pada TRAIN: 248825
Total paket label target pada TEST : 130834
Total paket label target          : 379659

SEMUA DISTRIBUSI LABEL TRAIN


,Label,Jumlah
0,Normal,954911
1,C_D,85466
2,P_I,64635
3,F_I,35112
4,M_F,33765
5,C_R,29847



SEMUA DISTRIBUSI LABEL TEST


,Label,Jumlah
0,Normal,660776
1,C_D,41203
2,C_R,29847
3,P_I,26013
4,F_I,16962
5,M_F,16809


In [4]:

# =========================
# HASIL
# =========================
print("\n===== HASIL ANALISIS PANJANG PAKET =====")

print(f"Panjang paket TERPANJANG : {max_length} byte")
print(f"Panjang paket TERPENDEK  : {min_length} byte")

print(f"\nNilai tengah min-max     : {middle_value} byte")
print(f"Median panjang paket     : {median_length} byte")

print(f"\nPanjang paling sering muncul : {most_common_length} byte")
print(f"Jumlah kemunculan            : {frequency} paket")


# =========================
# OPSIONAL:
# LIHAT 10 PANJANG TERBANYAK
# =========================
print("\n===== 10 PANJANG PALING SERING =====")

for length, count in length_counter.most_common(10):
    print(f"Panjang {length} byte -> {count} paket")


===== HASIL ANALISIS PANJANG PAKET =====
Panjang paket TERPANJANG : 434 byte
Panjang paket TERPENDEK  : 42 byte

Nilai tengah min-max     : 238.0 byte
Median panjang paket     : 60.0 byte

Panjang paling sering muncul : 60 byte
Jumlah kemunculan            : 1501026 paket

===== 10 PANJANG PALING SERING =====
Panjang 60 byte -> 1501026 paket
Panjang 434 byte -> 484973 paket
Panjang 90 byte -> 7583 paket
Panjang 68 byte -> 950 paket
Panjang 42 byte -> 568 paket
Panjang 82 byte -> 126 paket
Panjang 342 byte -> 90 paket
Panjang 384 byte -> 28 paket
Panjang 70 byte -> 1 paket
Panjang 242 byte -> 1 paket


In [5]:
# =========================

# JUMLAH NORMAL VS ATTACK

# =========================

train_labels = pd.read_csv(TRAIN_LABELS_PATH)
test_labels = pd.read_csv(TEST_LABELS_PATH)

# gabungkan label

all_labels = pd.concat(
[train_labels, test_labels],
ignore_index=True
)

# ambil kolom pertama

labels = all_labels.iloc[:, 0]

# hitung jumlah

normal_count = (labels == 0).sum()
attack_count = (labels == 1).sum()

total_count = len(labels)

# =========================

# OUTPUT

# =========================

print("\n===== DISTRIBUSI LABEL =====")

print(f"Total paket   : {total_count}")

print(f"\nNormal packet : {normal_count}")
print(f"Attack packet : {attack_count}")

print(f"\nPersentase Normal : {(normal_count / total_count) * 100:.2f}%")
print(f"Persentase Attack : {(attack_count / total_count) * 100:.2f}%")



===== DISTRIBUSI LABEL =====
Total paket   : 1995346

Normal packet : 0
Attack packet : 0

Persentase Normal : 0.00%
Persentase Attack : 0.00%
